# 8. Error Analysis

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from src.config.settings import load_config
from src.config.paths import (
    PROCESSED_DATA_DIR,
    ARTIFACTS_DIR,
    TUNED_MODELS_DIR
)
from src.data.load_data import load_raw_data
from src.models.training.threshold_tuning import tune_threshold, compute_threshold_curves
from sklearn.model_selection import StratifiedKFold, cross_val_predict

## 1. Load config

In [2]:
config = load_config()

TARGET = config["target"]
DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]
TUNING_CONFIG = config["tuning"]
THRESHOLD_CONFIG = config["threshold_tuning"]

print(f"Target:           {TARGET}")
print(f"Optimising for:   {THRESHOLD_CONFIG['metric']}")

Target:           good
Optimising for:   f1


## 2. Load processed data


In [3]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()

print(f"X_train shape: {X_train.shape}")

X_train shape: (1087, 11)


## 4. Load tuned models


In [4]:
tuned_models = joblib.load(ARTIFACTS_DIR / "tuned_models.pkl")

print(f"Available tuned models: {list(tuned_models.keys())}")

model_name = TUNING_CONFIG["models_to_tune"][0]
model = tuned_models[model_name]

print(f"Tuning threshold for: {model_name}")

Available tuned models: ['xgboost_classifier', 'logistic_regression']
Tuning threshold for: xgboost_classifier


In [5]:
import os
print("cores:", os.cpu_count())
print("model n_jobs:", model.get_params().get("n_jobs"))

cores: 16
model n_jobs: None


## 5. Model Score


In [9]:
%%time
from sklearn.base import clone

model_name = TUNING_CONFIG["models_to_tune"][0]
model = tuned_models[model_name]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_cv = clone(model).set_params(n_jobs=config["n_jobs"])
y_proba_train = cross_val_predict(model_cv, X_train, y_train, cv=cv,
                                  method="predict_proba")[:, 1]

CPU times: user 6.25 s, sys: 109 ms, total: 6.36 s
Wall time: 902 ms


In [14]:
df_error = pd.DataFrame({
    "true": y_train.values,
    "proba": y_proba_train,
})

print(df_error)


      true     proba
0        1  0.876027
1        1  0.806481
2        0  0.563307
3        1  0.912890
4        0  0.664497
...    ...       ...
1082     0  0.167173
1083     0  0.318041
1084     1  0.726824
1085     1  0.464941
1086     0  0.253239

[1087 rows x 2 columns]
